# Ingestão Silver: Bronze ➔ Silver com regras de Data Quality
Este notebook realiza, para cada micro-lote (arquivo) ainda não processado da Bronze:
- Leitura de `squad1/bronze/ecommerce_enderecos` filtrada por `bronze_source_file`.
- Aplicação das 10 regras de qualidade da tabela `ecommerce_enderecos` (Squad 1).
- Uma coluna booleana por regra: **`True` = a linha FALHOU na regra**, `False` = passou.
- Coluna `silver_processed_at` e carga Delta (`append`) em `squad1/silver/ecommerce_enderecos`.

| # | Coluna | Regra |
|---|--------|-------|
| 1 | `falha_r01_id_endereco_nulo_ou_duplicado` | `id_endereco` não pode ser nulo nem duplicado (no lote e na Silver) |
| 2 | `falha_r02_id_cliente_invalido` | `id_cliente` deve existir em `squad1/bronze/ecommerce_clientes` (FK) |
| 3 | `falha_r03_cep_invalido` | `cep` com exatamente 8 dígitos numéricos |
| 4 | `falha_r04_uf_invalida` | `estado` deve ser uma das 26 UFs + DF |
| 5 | `falha_r05_coordenadas_invalidas` | lat/long não nulas e dentro do Brasil |
| 6 | `falha_r06_principal_invalido` | exatamente 1 `is_principal = TRUE` por cliente (histórico do cliente) |
| 7 | `falha_r07_apelido_invalido` | `apelido` em `Casa`, `Trabalho`, `Outro` |
| 8 | `falha_r08_mais_de_3_enderecos` | no máximo 3 endereços por cliente (histórico do cliente) |
| 9 | `falha_r09_logradouro_vazio` | `logradouro` não nulo nem vazio |
| 10 | `falha_r10_numero_nulo` | `numero` não pode ser nulo |

As regras 6 e 8 consideram todos os endereços do cliente na Bronze até o micro-lote atual
(última versão de cada `id_endereco`), e não apenas o arquivo em processamento.

In [0]:
import os
from dotenv import load_dotenv
from pyspark.errors import AnalysisException
from pyspark.sql import Window
from pyspark.sql import functions as F, types as T

# 1. Carregamento de Parâmetros
load_dotenv(".env")

client_id            = os.getenv("ADLS_CLIENT_ID")
tenant_id            = os.getenv("ADLS_TENANT_ID")
client_secret        = os.getenv("ADLS_CLIENT_SECRET")
storage_account_name = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")

assert all([client_id, tenant_id, client_secret, storage_account_name]), "Faltou variável do ADLS no .env"

adls_opts = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

squad_base = f"abfss://squad1@{storage_account_name}.dfs.core.windows.net"
bronze_path = f"{squad_base}/bronze/ecommerce_enderecos"
silver_path = f"{squad_base}/silver/ecommerce_enderecos"
clientes_path = f"{squad_base}/bronze/ecommerce_clientes"

UFS_VALIDAS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA", "PB",
    "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]
APELIDOS_VALIDOS = ["Casa", "Trabalho", "Outro"]
LAT_MIN, LAT_MAX = -33.75, 5.27
LON_MIN, LON_MAX = -73.99, -32.39
MAX_ENDERECOS_POR_CLIENTE = 3

def ler_delta(path):
    try:
        df = spark.read.format("delta").options(**adls_opts).load(path)
        df.schema
        return df
    except AnalysisException as e:
        if any(m in str(e) for m in ("PATH_NOT_FOUND", "DELTA_PATH_DOES_NOT_EXIST", "DELTA_MISSING_DELTA_TABLE", "is not a Delta table")):
            return None
        raise

In [0]:
# 2. Definição das Regras de Data Quality
def historico_por_cliente(df_bronze, df_lote, source_file):
    w_versao = Window.partitionBy("id_endereco").orderBy(F.col("bronze_source_file").desc(), F.col("bronze_ingested_at").desc())
    return (
        df_bronze
        .filter((F.col("bronze_source_file") <= source_file) & F.col("id_endereco").isNotNull())
        .join(df_lote.select("id_cliente").distinct(), on="id_cliente", how="inner")
        .withColumn("_versao", F.row_number().over(w_versao))
        .filter(F.col("_versao") == 1)
        .groupBy("id_cliente")
        .agg(
            F.count("*").alias("_qtd_enderecos_cliente"),
            F.sum(F.when(F.col("is_principal") == True, 1).otherwise(0)).alias("_qtd_principais_cliente"),
        )
    )

def aplicar_regras_qualidade(df_lote, df_bronze, df_silver, source_file):
    df_clientes = ler_delta(clientes_path)
    if df_clientes is None:
        raise ValueError("Tabela squad1/bronze/ecommerce_clientes ausente (necessária para R02).")

    df = df_lote.withColumn("_qtd_id_lote", F.count("*").over(Window.partitionBy("id_endereco")))
    if df_silver is not None:
        ids_silver = df_silver.select("id_endereco").distinct().withColumn("_ja_na_silver", F.lit(True))
        df = df.join(ids_silver, on="id_endereco", how="left")
    else:
        df = df.withColumn("_ja_na_silver", F.lit(None).cast("boolean"))

    clientes = df_clientes.select("id_cliente").distinct().withColumn("_cliente_existe", F.lit(True))
    df = df.join(clientes, on="id_cliente", how="left")
    df = df.join(historico_por_cliente(df_bronze, df_lote, source_file), on="id_cliente", how="left")

    return (
        df
        .withColumn("falha_r01_id_endereco_nulo_ou_duplicado",
                    F.col("id_endereco").isNull() | (F.col("_qtd_id_lote") > 1) | F.coalesce(F.col("_ja_na_silver"), F.lit(False)))
        .withColumn("falha_r02_id_cliente_invalido",
                    F.col("id_cliente").isNull() | ~F.coalesce(F.col("_cliente_existe"), F.lit(False)))
        .withColumn("falha_r03_cep_invalido",
                    F.col("cep").isNull() | (F.length(F.regexp_replace(F.col("cep").cast("string"), "[^0-9]", "")) != 8))
        .withColumn("falha_r04_uf_invalida",
                    F.col("estado").isNull() | ~F.upper(F.trim(F.col("estado"))).isin(UFS_VALIDAS))
        .withColumn("falha_r05_coordenadas_invalidas",
                    F.col("latitude").isNull() | F.col("longitude").isNull()
                    | ~F.col("latitude").between(LAT_MIN, LAT_MAX)
                    | ~F.col("longitude").between(LON_MIN, LON_MAX))
        .withColumn("falha_r06_principal_invalido",
                    F.coalesce(F.col("_qtd_principais_cliente"), F.lit(0)) != 1)
        .withColumn("falha_r07_apelido_invalido",
                    F.col("apelido").isNull() | ~F.col("apelido").isin(APELIDOS_VALIDOS))
        .withColumn("falha_r08_mais_de_3_enderecos",
                    F.coalesce(F.col("_qtd_enderecos_cliente"), F.lit(0)) > MAX_ENDERECOS_POR_CLIENTE)
        .withColumn("falha_r09_logradouro_vazio",
                    F.col("logradouro").isNull() | (F.trim(F.col("logradouro")) == ""))
        .withColumn("falha_r10_numero_nulo", F.col("numero").isNull())
        .drop("_qtd_id_lote", "_ja_na_silver", "_cliente_existe", "_qtd_enderecos_cliente", "_qtd_principais_cliente")
    )

def arquivos_pendentes(df_bronze, df_silver):
    bronze_files = {r["bronze_source_file"] for r in df_bronze.select("bronze_source_file").distinct().collect()}
    silver_files = set()
    if df_silver is not None:
        silver_files = {r["bronze_source_file"] for r in df_silver.select("bronze_source_file").distinct().collect()}
    return sorted(bronze_files - silver_files)

In [0]:
# 3. Execução Pontual da Silver e Retorno
df_bronze = ler_delta(bronze_path)
df_silver = ler_delta(silver_path)

pendentes = [] if df_bronze is None else arquivos_pendentes(df_bronze, df_silver)
print(f"Arquivos pendentes para Silver: {len(pendentes)}")

sucessos = 0
for source_file in pendentes:
    try:
        print(f"Processando Silver para: {source_file}")
        df_lote = (
            df_bronze
            .filter(F.col("bronze_source_file") == source_file)
            .drop("ano", "mes", "dia", "hora")
        )

        df_resultado = (
            aplicar_regras_qualidade(df_lote, df_bronze, df_silver, source_file)
            .withColumn("silver_processed_at", F.current_timestamp())
        )

        df_resultado.write.format("delta").mode("append").options(**adls_opts).save(silver_path)
        df_silver = ler_delta(silver_path)
        sucessos += 1
        print("✔ Sucesso na Silver.")
    except Exception as e:
        print(f"✖ Falha em '{source_file}': {e}")

dbutils.notebook.exit(str(sucessos))